# Phases 3-6 validation: effective potential, shell filling, SCF loop, total energy

Checks, in order:
1. `potentials.hartree_potential` reproduces the exact hydrogen 1s self-Coulomb energy (5/16 Ha).
2. `shells.ground_state_configuration` matches known reference configurations, including Madelung exceptions.
3. `scf.run_scf` converges for closed-shell atoms (He, Ne, Ar) and Madelung-exception atoms (Cr, Cu), landing within ~0.2-3.5% of literature non-relativistic HF energies (expected gap: this is Xa local exchange, not exact HF).
4. SCF convergence curves and the radial density's shell-peak structure, visually.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid

import potentials as pot
import shells
import scf
import hydrogenic as hy

## 1. Hartree potential: exact hydrogen 1s self-Coulomb energy

In [ ]:
r = np.linspace(1e-6, 40, 200000)
R10 = hy.radial_wavefunction(1, 0, r, Z=1)
rho_1s = R10**2 / (4 * np.pi)

V_H = pot.hartree_potential(r, rho_1s)
E_H = 0.5 * trapezoid(V_H * rho_1s * 4 * np.pi * r**2, r)
print(f'E_H (numeric)      = {E_H:.8f}')
print(f'E_H (exact, 5/16)  = {5/16:.8f}')
print(f'V_H(r)*r at r=40 (should -> N=1): {V_H[-1]*r[-1]:.8f}')

## 2. Shell filling: known configurations, including Madelung exceptions

In [ ]:
for Z, name in [(26, 'Fe'), (24, 'Cr'), (29, 'Cu'), (46, 'Pd'), (57, 'La'), (64, 'Gd'), (92, 'U')]:
    cfg = shells.ground_state_configuration(Z)
    print(f'{name:3s} (Z={Z:3d}, N={sum(cfg.values()):3d}): {shells.format_configuration(cfg)}')

## 3. Full SCF runs

He/Ne/Ar (closed shell) and Cr/Cu (Madelung exceptions, `mix_beta` lowered for the harder open-shell case). Literature non-relativistic HF energies for comparison: He -2.86168 Ha, Ne -128.547 Ha, Ar -526.818 Ha (no simple literature Xa reference for Cr/Cu at this alpha).

In [ ]:
results = {}
for Z, name, kwargs in [
    (2, 'He', {}),
    (10, 'Ne', {}),
    (18, 'Ar', {}),
    (24, 'Cr', {'mix_beta': 0.2}),
    (29, 'Cu', {'mix_beta': 0.2}),
]:
    res = scf.run_scf(Z, max_iter=200, **kwargs)
    results[name] = res
    Q = trapezoid(4 * np.pi * res['r']**2 * res['rho'], res['r'])
    print(f'{name:3s} (Z={Z:3d}): E_total={res["E_total"]:12.5f} Ha  iters={res["iterations"]:4d}  N_check={Q:.6f}  config={shells.format_configuration(res["config"])}')

## 4. Convergence curves and radial density shell structure

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

for name in ['He', 'Ar', 'Cr', 'Cu']:
    h = results[name]['history']
    axs[0].plot(range(1, len(h) + 1), h, marker='.', label=name)
axs[0].set_xlabel('SCF iteration')
axs[0].set_ylabel('E_total (Ha)')
axs[0].set_yscale('symlog')
axs[0].set_title('SCF convergence')
axs[0].legend()

for name, rmax in [('Ar', 3.0), ('Cr', 3.0)]:
    res = results[name]
    r = res['r']
    mask = r < rmax
    axs[1].plot(r[mask], 4 * np.pi * r[mask]**2 * res['rho'][mask], label=name)
axs[1].set_xlabel('r (Bohr)')
axs[1].set_ylabel(r'$4\pi r^2 \rho(r)$')
axs[1].set_title('Radial density (shell peaks)')
axs[1].legend()

plt.tight_layout()
fig.savefig('media/phase3_6_scf_validation.png', dpi=110)
plt.show()